# NEU-ESC: Phân loại cảm xúc bằng RNN truyền thống

**Mục tiêu:** dùng `SimpleRNN` cho đúng 4 nhãn: Neutral, Positive, Negative, Toxic.
Notebook lấy phân tầng **30% từng split gốc**: 6.914 train, 992 validation, 1.984 test.
Comment và giải thích bằng tiếng Việt; code chia thành các bước nhỏ để cả team đọc được.

## Cách chạy trên Google Colab A100
1. Upload notebook này lên Colab.
2. Chọn **Runtime → Change runtime type → Python 3 → A100 GPU**.
3. Upload `NEU.zip` vào bảng **Files** bên trái, hoặc dùng ô upload ở bước 2.
4. Chạy từ trên xuống. Giữ cấu hình mặc định cho lần chạy đầu tiên.
5. Cuối notebook, tải file ZIP kết quả về máy trước khi ngắt Colab.

Không cần Hugging Face token hoặc tải dataset từ Internet. Notebook không tự cài lại
TensorFlow/CUDA của Colab. Nếu máy thiếu thư viện, đọc hướng dẫn kèm notebook.

**Giới hạn:** dừng sớm theo validation loss và đặt ngân sách train 25 phút để chừa thời gian
đánh giá. Chưa có số đo A100 trước khi bạn chạy; thời gian thực tế sẽ được lưu lại.
Không thay cấu hình sau khi xem test rồi coi test đó là một đánh giá độc lập mới.

## 1. Thư viện, GPU và cấu hình

Tất cả tham số cần chỉnh nằm trong `CONFIG`. `SAMPLE_FRACTION=0.20` là phương án
dự phòng nếu thật sự cần giảm dữ liệu. Khi đổi cấu hình, chạy lại notebook từ đầu.
Không tăng dữ liệu chỉ để dùng hết bộ nhớ A100.

Dùng `float32` thay vì mixed precision để pipeline dễ hiểu và hạn chế thêm vấn đề số học.
Seed giúp tái lập lựa chọn mẫu; không cam kết kết quả GPU giống từng bit giữa các môi trường.

In [ ]:
import hashlib
import importlib.metadata
import json
import math
import platform
import shutil
import time
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from tensorflow import keras
from tensorflow.keras import layers

CONFIG = {
    "SEED": 42,
    "SAMPLE_FRACTION": 0.30,
    "MAX_TOKENS": 10_000,
    "MAX_LENGTH": 64,
    "EMBEDDING_DIM": 64,
    "RNN_UNITS": 64,
    "DROPOUT": 0.20,
    "BATCH_SIZE": 64,
    "MAX_EPOCHS": 6,
    "LEARNING_RATE": 0.001,
    "PATIENCE": 2,
    "TRAIN_BUDGET_MINUTES": 25,
}
LABEL_IDS = [0, 1, 2, 3]
LABEL_NAMES = ["Neutral", "Positive", "Negative", "Toxic"]
LABEL_VI = ["Trung tính", "Tích cực", "Tiêu cực", "Độc hại"]

keras.utils.set_random_seed(CONFIG["SEED"])
keras.mixed_precision.set_global_policy("float32")
gpu_devices = tf.config.list_physical_devices("GPU")
gpu_info = [tf.config.experimental.get_device_details(gpu) for gpu in gpu_devices]
print("TensorFlow:", tf.__version__)
print("GPU:", gpu_info if gpu_info else "Không có GPU; đang dùng CPU")
if not gpu_devices:
    print("Trên Colab: hãy chọn runtime A100 rồi chạy lại từ đầu.")

# Mỗi lần chạy dùng thư mục riêng để không nạp nhầm checkpoint cũ.
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
RUN_DIR = Path("neu_rnn_outputs") / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Thư mục kết quả:", RUN_DIR.resolve())


## 2. Đọc NEU.zip và kiểm tra dữ liệu

Đặt ZIP cạnh notebook hoặc sửa `ZIP_PATH` nếu dùng Google Drive đã mount.
Chỉ đọc ba CSV trực tiếp từ ZIP, không cần giải nén toàn bộ.
Nhãn `0.0`…`3.0` phải được kiểm tra trước khi đổi sang số nguyên.
Nếu có dữ liệu lỗi hoặc trùng, notebook dừng để kiểm tra thay vì âm thầm thay đổi split.

In [ ]:
ZIP_PATH = Path("NEU.zip")

if not ZIP_PATH.is_file():
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError("Đặt NEU.zip cạnh notebook hoặc sửa ZIP_PATH.") from error
    print("Chọn file NEU.zip từ máy của bạn.")
    files.upload()
    if not ZIP_PATH.is_file():
        raise FileNotFoundError("Chưa thấy NEU.zip. Kiểm tra tên file hoặc sửa ZIP_PATH.")

zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
original_data = {}
with zipfile.ZipFile(ZIP_PATH) as archive:
    for split in ["train", "val", "test"]:
        with archive.open(f"NEU/{split}_set.csv") as csv_file:
            frame = pd.read_csv(csv_file, encoding="utf-8-sig")
        if not {"text", "sentiment"}.issubset(frame.columns):
            raise ValueError(f"{split}: thiếu cột text hoặc sentiment.")
        labels = pd.to_numeric(frame["sentiment"], errors="coerce")
        if not labels.isin(LABEL_IDS).all():
            raise ValueError(f"{split}: nhãn không hợp lệ hoặc bị thiếu.")
        if not frame["text"].map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"{split}: có văn bản thiếu hoặc không phải chuỗi.")

        # row_id là vị trí dòng dữ liệu, bắt đầu từ 0 và không tính header CSV.
        frame = frame[["text"]].copy()
        frame["label"] = labels.astype("int32")
        frame["row_id"] = np.arange(len(frame))
        original_data[split] = frame

print("SHA-256 của ZIP:", zip_sha256)
print("Tổng số mẫu:", sum(len(frame) for frame in original_data.values()))


In [ ]:
import unicodedata


def normalize_text(text):
    """Giữ dấu, emoji và từ phủ định; chỉ chuẩn hóa Unicode, chữ và khoảng trắng."""
    if not isinstance(text, str):
        raise TypeError("Văn bản phải là chuỗi.")
    text = unicodedata.normalize("NFC", text)
    return " ".join(text.lower().split())

for split, frame in original_data.items():
    frame["clean_text"] = frame["text"].map(normalize_text)
    if frame["clean_text"].eq("").any():
        raise ValueError(f"{split}: có câu rỗng sau chuẩn hóa.")
    if frame["clean_text"].duplicated().any():
        raise ValueError(f"{split}: có câu trùng, cần xem lại trước khi lấy mẫu.")
    if set(frame["label"]) != set(LABEL_IDS):
        raise ValueError(f"{split}: phải có đủ bốn nhãn.")

for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = set(original_data[left]["clean_text"]) & set(original_data[right]["clean_text"])
    if overlap:
        raise ValueError(f"Có {len(overlap)} câu trùng giữa {left} và {right}.")

original_counts = pd.DataFrame({
    split: frame["label"].value_counts().reindex(LABEL_IDS, fill_value=0)
    for split, frame in original_data.items()
}).T
original_counts.columns = LABEL_NAMES
original_counts["Total"] = original_counts.sum(axis=1)
display(original_counts)
original_counts.to_csv(RUN_DIR / "original_counts.csv", encoding="utf-8-sig")
print("Đã kiểm tra: đủ 4 nhãn, không có câu rỗng hoặc trùng hoàn toàn.")


## 3. Lấy mẫu phân tầng, giữ tỷ lệ gốc

Lấy cùng tỷ lệ trong **mỗi split**, không gộp và chia lại. Trong từng split, chia số lượng
theo tỷ lệ nhãn gốc và phân bổ phần dư làm tròn. Không cân bằng bằng cách nhân bản mẫu Toxic.
Với ZIP đã cung cấp và tỷ lệ 30%, kết quả phải là 6.914 / 992 / 1.984 mẫu.

In [ ]:
def stratified_subset(frame, fraction, seed):
    """Chọn ngẫu nhiên trong mỗi lớp, giữ tỷ lệ lớp gần với dữ liệu gốc."""
    if not 0 < fraction <= 1:
        raise ValueError("Tỷ lệ lấy mẫu phải lớn hơn 0 và không vượt quá 1.")
    target_size = round(len(frame) * fraction)
    counts = frame["label"].value_counts().reindex(LABEL_IDS, fill_value=0)
    exact_counts = target_size * counts / len(frame)
    quotas = np.floor(exact_counts).astype(int)

    # Ưu tiên lớp có phần dư lớn nhất; nếu bằng nhau thì dùng mã nhãn nhỏ hơn.
    remainder_order = sorted(
        LABEL_IDS,
        key=lambda label: (-(exact_counts[label] - quotas[label]), label),
    )
    for label in remainder_order[:target_size - int(quotas.sum())]:
        quotas[label] += 1
    if (quotas == 0).any():
        raise ValueError("Subset quá nhỏ để giữ đủ bốn lớp; hãy tăng tỷ lệ.")

    pieces = []
    for label in LABEL_IDS:
        class_rows = frame[frame["label"] == label]
        chosen = class_rows.sample(n=int(quotas[label]), random_state=seed + label)
        pieces.append(chosen)
    subset = pd.concat(pieces).sample(frac=1, random_state=seed).reset_index(drop=True)
    assert len(subset) == target_size
    assert subset["row_id"].is_unique
    return subset

data = {
    split: stratified_subset(frame, CONFIG["SAMPLE_FRACTION"], CONFIG["SEED"])
    for split, frame in original_data.items()
}
train_df, val_df, test_df = data["train"], data["val"], data["test"]

subset_counts = pd.DataFrame({
    split: frame["label"].value_counts().reindex(LABEL_IDS, fill_value=0)
    for split, frame in data.items()
}).T
subset_counts.columns = LABEL_NAMES
subset_counts["Total"] = subset_counts.sum(axis=1)
display(subset_counts)
subset_counts.to_csv(RUN_DIR / "subset_counts.csv", encoding="utf-8-sig")

manifest = pd.concat([
    frame[["row_id", "label"]].assign(split=split)
    for split, frame in data.items()
], ignore_index=True)
manifest.to_csv(RUN_DIR / "sample_manifest.csv", index=False)

ratio_difference = (
    subset_counts[LABEL_NAMES].div(subset_counts["Total"], axis=0)
    - original_counts[LABEL_NAMES].div(original_counts["Total"], axis=0)
) * 100
print("Chênh lệch tỷ lệ so với split gốc, đơn vị điểm phần trăm:")
display(ratio_difference.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
subset_counts[LABEL_NAMES].plot.bar(ax=axes[0], rot=0)
axes[0].set(title="Số mẫu theo nhãn trong subset", ylabel="Số mẫu")

train_lengths = train_df["clean_text"].str.split().str.len()
axes[1].hist(train_lengths.clip(upper=160), bins=35)
axes[1].axvline(CONFIG["MAX_LENGTH"], color="red", linestyle="--", label="Giới hạn token")
axes[1].set(title="Độ dài câu train (gom độ dài >160 vào 160)", xlabel="Số token")
axes[1].legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "dataset_overview.png", dpi=160)
plt.show()
print(f"Câu train bị cắt: {(train_lengths > CONFIG['MAX_LENGTH']).mean():.2%}")
print("Toxic trong validation/test:", int(subset_counts.loc["val", "Toxic"]),
      int(subset_counts.loc["test", "Toxic"]), "— F1 lớp này có thể dao động do ít mẫu.")


## 4. Vocabulary và chuỗi token

Chỉ `adapt` vocabulary bằng **subset train**. `standardize=None` vì văn bản đã được
chuẩn hóa bằng hàm dùng chung; giữ emoji, dấu và từ phủ định. Tách theo khoảng trắng
đơn giản nhưng chưa phải tách từ tiếng Việt đầy đủ.

Vectorize một lần trên CPU, sau đó đưa các số nguyên lên GPU để train. Index 0 là padding,
index 1 là OOV. Câu dài giữ 64 token đầu; padding nằm phía phải.

In [ ]:
with tf.device("/CPU:0"):
    vectorizer = layers.TextVectorization(
        max_tokens=CONFIG["MAX_TOKENS"],
        standardize=None,
        split="whitespace",
        output_mode="int",
        output_sequence_length=CONFIG["MAX_LENGTH"],
        name="text_vectorizer",
    )
    vocabulary_data = tf.data.Dataset.from_tensor_slices(
        train_df["clean_text"].tolist()
    ).batch(256)
    vectorizer.adapt(vocabulary_data)

    def vectorize(frame):
        texts = tf.constant(frame["clean_text"].tolist(), dtype=tf.string)
        return vectorizer(texts).numpy().astype("int32")

    x_train = vectorize(train_df)
    x_val = vectorize(val_df)
    x_test = vectorize(test_df)

y_train = train_df["label"].to_numpy(dtype="int32")
y_val = val_df["label"].to_numpy(dtype="int32")
y_test = test_df["label"].to_numpy(dtype="int32")
vocabulary = vectorizer.get_vocabulary()
print("Kích thước vocabulary thực tế:", len(vocabulary))
print("Shape train/validation/test:", x_train.shape, x_val.shape, x_test.shape)
assert vocabulary[0] == "" and vocabulary[1] == "[UNK]"
assert (x_train != 0).any(axis=1).all()


## 5. Class weight và batch

Class weight tăng đóng góp của lớp ít mẫu vào **loss**, không thay đổi tỷ lệ dữ liệu.
Loss train có trọng số còn validation loss không có trọng số; không so sánh trực tiếp
độ cao hai đường loss như thể chúng có cùng cách tính. Accuracy bên dưới là accuracy
không trọng số. Không học hoặc tính trọng số bằng test.

In [ ]:
train_class_counts = np.bincount(y_train, minlength=4)
class_weight = {
    label: float(len(y_train) / (4 * train_class_counts[label]))
    for label in LABEL_IDS
}
print("Class weight:", class_weight)

def make_dataset(x, y, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        dataset = dataset.shuffle(len(x), seed=CONFIG["SEED"], reshuffle_each_iteration=True)
    return dataset.batch(CONFIG["BATCH_SIZE"]).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val)
print("Số batch train mỗi epoch:", math.ceil(len(x_train) / CONFIG["BATCH_SIZE"]))


## 6. SimpleRNN truyền thống và vanishing gradient

`h_t = tanh(W_x x_t + W_h h_(t-1) + b)` — không có gate của LSTM/GRU.

- **64 token:** rút ngắn đường truyền gradient theo thời gian.
- **Khởi tạo trực giao:** giúp ổn định phép nhân với ma trận hồi quy lúc khởi tạo;
  đạo hàm `tanh` vẫn có thể làm gradient nhỏ dần.
- **Masked average pooling:** loss có đường trực tiếp tới hidden state ở từng vị trí,
  giảm phụ thuộc vào hidden state cuối. Padding không được tham gia trung bình.
- **Một tầng RNN:** giữ mô hình gọn. Dropout sau pooling nhằm giảm overfitting.

Các biện pháp này **không loại bỏ hoàn toàn** vanishing gradient và không chứng minh
giảm một tỷ lệ phần trăm cụ thể. `clipnorm=1.0` chỉ hạn chế **exploding gradient**.

In [ ]:
token_input = keras.Input(shape=(None,), dtype="int32", name="token_ids")
embedding_layer = layers.Embedding(
    input_dim=len(vocabulary),
    output_dim=CONFIG["EMBEDDING_DIM"],
    mask_zero=True,
    name="embedding",
)
rnn_layer = layers.SimpleRNN(
    CONFIG["RNN_UNITS"],
    activation="tanh",
    recurrent_initializer="orthogonal",
    return_sequences=True,
    recurrent_dropout=0.0,
    name="simple_rnn",
)

hidden_states = rnn_layer(embedding_layer(token_input))
sentence_vector = layers.GlobalAveragePooling1D(name="masked_average")(hidden_states)
sentence_vector = layers.Dropout(CONFIG["DROPOUT"], name="dropout")(sentence_vector)
probabilities = layers.Dense(4, activation="softmax", name="sentiment")(sentence_vector)
model = keras.Model(token_input, probabilities, name="neu_simple_rnn")

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=CONFIG["LEARNING_RATE"], clipnorm=1.0
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    jit_compile=False,
)
model.summary()


### Kiểm tra padding trước khi train

Cùng một câu có thêm padding phải cho cùng kết quả khi tắt dropout. Đây là kiểm tra
masking của cả RNN và pooling, không phải kiểm tra chất lượng dự đoán.

In [ ]:
short_example = x_train[0][x_train[0] != 0][:8]
without_padding = short_example[np.newaxis, :]
with_padding = np.pad(without_padding, ((0, 0), (0, 7)))
np.testing.assert_allclose(
    model(without_padding, training=False).numpy(),
    model(with_padding, training=False).numpy(),
    rtol=1e-5, atol=1e-6,
)
print("OK: padding không làm thay đổi dự đoán.")


## 7. Train, lưu checkpoint và kiểm soát thời gian

Epoch đầu cũng là epoch train thật. Callback đo thời gian, ước tính tổng thời gian và
dừng nếu chạm ngân sách, hoặc không còn đủ thời gian dự kiến cho một epoch tiếp theo.
Thời gian đo gồm train và validation trong `fit`, không gồm toàn bộ các cell chuẩn bị.

Kiểm tra ngân sách diễn ra tại cuối batch/epoch nên vẫn có thể vượt một ít do batch đang
chạy, validation hoặc lưu checkpoint. Mốc 25 phút chừa dư địa so với mục tiêu 30 phút.
Nếu rerun cell này, đó là một lần train khác; thời gian các lần chạy phải được cộng khi báo cáo.

In [ ]:
class TrainingTimer(keras.callbacks.Callback):
    """Dừng theo thời gian thực và lưu thời lượng từng epoch."""

    def __init__(self, budget_minutes):
        super().__init__()
        self.budget_seconds = budget_minutes * 60
        self.epoch_seconds = []
        self.stop_reason = ""

    def on_train_begin(self, logs=None):
        self.started_at = time.perf_counter()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_started_at = time.perf_counter()

    def on_train_batch_end(self, batch, logs=None):
        if time.perf_counter() - self.started_at >= self.budget_seconds:
            if not self.stop_reason:
                print("\nĐã chạm ngân sách; dừng train và đánh giá validation của epoch này.")
            self.stop_reason = "Đạt ngân sách thời gian"
            self.model.stop_training = True

    def on_epoch_end(self, epoch, logs=None):
        duration = time.perf_counter() - self.epoch_started_at
        self.epoch_seconds.append(duration)
        remaining = self.budget_seconds - (time.perf_counter() - self.started_at)
        print(f"Epoch {epoch + 1}: {duration:.1f}s; ngân sách còn {max(0, remaining):.1f}s")
        if epoch == 0:
            print(f"Ước lượng thô cho {CONFIG['MAX_EPOCHS']} epoch: "
                  f"{duration * CONFIG['MAX_EPOCHS'] / 60:.1f} phút (epoch đầu có chi phí khởi tạo).")
        if remaining < max(duration * 1.15, 30) and not self.model.stop_training:
            self.stop_reason = "Không đủ ngân sách dự kiến cho epoch tiếp theo"
            self.model.stop_training = True
            print(self.stop_reason)


# Tạo model mới cho mỗi lần chạy cell này; không vô tình train tiếp trọng số cũ.
keras.utils.set_random_seed(CONFIG["SEED"])
model = keras.models.clone_model(model)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG["LEARNING_RATE"], clipnorm=1.0),
    loss="sparse_categorical_crossentropy",
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    jit_compile=False,
)
fit_id = datetime.now().strftime("fit_%H%M%S_%f")
checkpoint_path = RUN_DIR / f"{fit_id}.weights.h5"
timer = TrainingTimer(CONFIG["TRAIN_BUDGET_MINUTES"])
callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(checkpoint_path), monitor="val_loss", mode="min",
        save_best_only=True, save_weights_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=CONFIG["PATIENCE"],
        restore_best_weights=True,
    ),
    keras.callbacks.TerminateOnNaN(),
    timer,
]

fit_started_at = time.perf_counter()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["MAX_EPOCHS"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=2,
)
train_seconds = time.perf_counter() - fit_started_at
if not checkpoint_path.is_file():
    raise RuntimeError("Chưa có checkpoint hợp lệ. Kiểm tra loss và log trước khi đánh giá.")
if not all(np.isfinite(history.history[key]).all() for key in ["loss", "val_loss"]):
    raise RuntimeError("Loss train/validation có NaN/Inf. Kiểm tra trước khi dùng kết quả.")

# Luôn nạp checkpoint tốt nhất, kể cả khi dừng vì thời gian thay vì early stopping.
model.load_weights(checkpoint_path)
best_epoch = int(np.nanargmin(history.history["val_loss"])) + 1
print(f"Train thực tế: {train_seconds / 60:.2f} phút; checkpoint tốt nhất: epoch {best_epoch}.")


## 8. Xem validation và đường học

Chỉ sử dụng validation để xem xét mô hình trước khi chốt. Nếu thay cấu hình, chạy lại từ đầu.
Sau khi chạy bước test, không dùng điểm test để chọn thêm siêu tham số.

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
history_df["seconds"] = timer.epoch_seconds
history_df.to_csv(RUN_DIR / "training_history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["loss"], label="Train (có class weight)")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation (không weight)")
axes[0].set(title="Loss: hai đường có cách tính trọng số khác nhau", xlabel="Epoch")
axes[1].plot(history_df["epoch"], history_df["accuracy"], label="Train")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="Validation")
axes[1].set(title="Accuracy không trọng số", xlabel="Epoch")
for axis in axes:
    axis.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=160)
plt.show()

val_probabilities = model.predict(x_val, batch_size=CONFIG["BATCH_SIZE"], verbose=0)
val_predictions = val_probabilities.argmax(axis=1)
validation_metrics = {
    "accuracy": float(accuracy_score(y_val, val_predictions)),
    "macro_f1": float(f1_score(y_val, val_predictions, labels=LABEL_IDS, average="macro", zero_division=0)),
}
print("Validation:", validation_metrics)
print(classification_report(y_val, val_predictions, labels=LABEL_IDS,
                            target_names=LABEL_NAMES, zero_division=0, digits=4))


## 9. Đánh giá cuối trên test và so sánh baseline

Chạy sau khi đã chốt mô hình. Baseline luôn đoán lớp phổ biến nhất **trong train**.
Macro-F1 là chỉ số chính; test ở đây là subset, không phải toàn bộ test NEU-ESC.

In [ ]:
test_probabilities = model.predict(x_test, batch_size=CONFIG["BATCH_SIZE"], verbose=0)
test_predictions = test_probabilities.argmax(axis=1)
majority_label = int(np.argmax(train_class_counts))
baseline_predictions = np.full(len(y_test), majority_label, dtype="int32")

def calculate_metrics(y_true, y_pred):
    class_f1 = f1_score(y_true, y_pred, labels=LABEL_IDS, average=None, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(class_f1.mean()),
        "f1_toxic": float(class_f1[3]),
    }

baseline_metrics = calculate_metrics(y_test, baseline_predictions)
test_metrics = calculate_metrics(y_test, test_predictions)
comparison = pd.DataFrame([
    {"model": "Majority baseline", **baseline_metrics},
    {"model": "SimpleRNN + masked average", **test_metrics},
])
display(comparison)
comparison.to_csv(RUN_DIR / "model_comparison.csv", index=False)

report = classification_report(
    y_test, test_predictions, labels=LABEL_IDS,
    target_names=LABEL_NAMES, output_dict=True, zero_division=0,
)
display(pd.DataFrame(report).T)
pd.DataFrame(report).T.to_csv(RUN_DIR / "classification_report.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=LABEL_IDS, display_labels=LABEL_NAMES,
    cmap="Blues", values_format="d", ax=ax, colorbar=False,
)
ax.set_title("Confusion matrix — test subset NEU-ESC")
fig.tight_layout()
fig.savefig(RUN_DIR / "confusion_matrix.png", dpi=160)
plt.show()


In [ ]:
test_results = test_df[["row_id", "text", "label"]].copy()
test_results["true_label"] = [LABEL_NAMES[label] for label in y_test]
test_results["predicted_label"] = [LABEL_NAMES[label] for label in test_predictions]
for label, name in enumerate(LABEL_NAMES):
    test_results[f"prob_{name}"] = test_probabilities[:, label]
test_results.to_csv(RUN_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")

errors = test_results[test_results["true_label"] != test_results["predicted_label"]]
print(f"Số câu dự đoán sai: {len(errors)}/{len(test_results)}")
display(errors[["text", "true_label", "predicted_label"]].head(5))
print("Đọc lỗi để báo cáo hạn chế; không sửa mô hình theo test rồi báo lại như test độc lập.")


## 10. Lưu model có vocabulary và kiểm tra nạp lại

File `model.keras` gồm vectorizer + RNN. Đầu vào của file này là **chuỗi đã chuẩn hóa**;
hàm chuẩn hóa được lưu riêng trong `preprocessing.py`. Không cần custom layer hoặc custom loss.
Ứng dụng phải từ chối câu rỗng trước khi gọi model: pooling trên toàn padding không có ý nghĩa.

Xuất cấu hình, môi trường, ánh xạ nhãn, manifest chọn mẫu và số liệu thật để viết báo cáo.

In [ ]:
clean_text_input = keras.Input(shape=(), dtype=tf.string, name="normalized_text")
inference_output = model(vectorizer(clean_text_input))
inference_model = keras.Model(clean_text_input, inference_output, name="neu_sentiment_inference")
model_path = RUN_DIR / "model.keras"
inference_model.save(model_path)
loaded_model = keras.models.load_model(model_path, compile=False)

probe_texts = tf.constant(val_df["clean_text"].head(8).tolist())
original_probabilities = model(x_val[:8], training=False).numpy()
reloaded_probabilities = loaded_model(probe_texts, training=False).numpy()
np.testing.assert_allclose(original_probabilities, reloaded_probabilities, rtol=1e-5, atol=1e-6)
print("OK: model sau lưu/nạp cho kết quả khớp model đã đánh giá.")

def save_json(filename, value):
    (RUN_DIR / filename).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

environment = {"python": platform.python_version(), "gpu": gpu_info}
for package in ["tensorflow", "keras", "numpy", "pandas", "scikit-learn", "matplotlib"]:
    environment[package] = importlib.metadata.version(package)

save_json("environment.json", environment)
save_json("config.json", {
    **CONFIG,
    "actual_vocabulary_size": len(vocabulary),
    "class_weight": class_weight,
    "normalization": "Unicode NFC -> lowercase -> collapse whitespace",
    "truncation": "Keep the first MAX_LENGTH whitespace tokens; right padding",
    "source_zip_sha256": zip_sha256,
    "row_id_definition": "Zero-based data row in the original split CSV, excluding header",
    "subset_sizes": {split: len(frame) for split, frame in data.items()},
})
save_json("label_map.json", {
    str(label): {"name": LABEL_NAMES[label], "vietnamese": LABEL_VI[label]}
    for label in LABEL_IDS
})
save_json("vocabulary.json", vocabulary)
save_json("metrics.json", {
    "validation": validation_metrics,
    "test": test_metrics,
    "baseline_test": baseline_metrics,
    "test_support": np.bincount(y_test, minlength=4).tolist(),
    "train_seconds": train_seconds,
    "epochs_run": len(history_df),
    "best_epoch_by_val_loss": best_epoch,
    "epoch_seconds": timer.epoch_seconds,
    "time_budget_stop_reason": timer.stop_reason,
    "evaluation_scope": "Stratified subset of the official NEU-ESC test split",
})


In [ ]:
# Lưu cùng hàm chuẩn hóa dùng trong notebook để tái sử dụng khi làm demo.
preprocessing_source = '''
import unicodedata


def normalize_text(text):
    """Giữ dấu, emoji và từ phủ định; chỉ chuẩn hóa Unicode, chữ và khoảng trắng."""
    if not isinstance(text, str):
        raise TypeError("Văn bản phải là chuỗi.")
    text = unicodedata.normalize("NFC", text)
    return " ".join(text.lower().split())
'''
predict_source = '''
"""Suy luận bằng artifact đã tải về; không cần đọc lại dữ liệu train."""
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from preprocessing import normalize_text

ROOT = Path(__file__).resolve().parent
model = tf.keras.models.load_model(ROOT / "model.keras", compile=False)
label_map = json.loads((ROOT / "label_map.json").read_text(encoding="utf-8"))
config = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))


def predict_text(text):
    clean_text = normalize_text(text)
    if not clean_text:
        raise ValueError("Hãy nhập một bình luận không rỗng.")
    scores = model(tf.constant([clean_text]), training=False).numpy()[0]
    label = int(np.argmax(scores))
    return {
        "label": label_map[str(label)]["vietnamese"],
        "truncated": len(clean_text.split()) > config["MAX_LENGTH"],
        "probabilities": {label_map[str(i)]["name"]: float(p) for i, p in enumerate(scores)},
    }


if __name__ == "__main__":
    print(predict_text(input("Nhập bình luận: ")))
'''
artifact_readme = '''
# Kết quả NEU-ESC SimpleRNN

- model.keras: vectorizer và SimpleRNN đã chọn bằng validation loss.
- preprocessing.py + predict.py: suy luận từ văn bản gốc, từ chối câu rỗng.
- metrics.json và các CSV/PNG: số liệu, biểu đồ và thời gian chạy thực tế.
- sample_manifest.csv: chỉ số dòng (bắt đầu từ 0, không tính header) trong từng CSV gốc.
- config.json: cấu hình, SHA-256 ZIP nguồn, tỷ lệ lấy mẫu và quy tắc tiền xử lý.
- requirements-inference.txt: phiên bản thư viện cho suy luận ở môi trường hiện tại.

Chạy lại suy luận trong môi trường Python tương thích:
    pip install -r requirements-inference.txt
    python predict.py

Đầu vào gọi trực tiếp model.keras phải được normalize_text() trước và không được rỗng.
Model giữ các token đầu câu theo MAX_LENGTH; các giá trị Softmax không phải độ chính xác
đã hiệu chuẩn của từng câu. Không dùng model để tự động quyết định xử lý một người.

Điểm đánh giá là trên subset của test NEU-ESC. Lớp Toxic có ít mẫu; xem support trong report.
Class weight chỉ dùng khi train. Không diễn giải clipping là cách chữa vanishing gradient.
Không so sánh điểm trên subset này như benchmark toàn bộ test gốc.

Nguồn: https://huggingface.co/datasets/hung20gg/NEU-ESC
File test_predictions.csv chứa văn bản nguồn; kiểm tra quyền chia sẻ trước khi công khai.
'''

(RUN_DIR / "preprocessing.py").write_text(preprocessing_source, encoding="utf-8")
(RUN_DIR / "predict.py").write_text(predict_source, encoding="utf-8")
(RUN_DIR / "README.md").write_text(artifact_readme, encoding="utf-8")
requirements = "\n".join(
    f"{package}=={environment[package]}" for package in ["tensorflow", "keras", "numpy"]
)
(RUN_DIR / "requirements-inference.txt").write_text(requirements + "\n", encoding="utf-8")
print("Đã lưu artifact vào:", RUN_DIR.resolve())


## 11. Thử dự đoán trong notebook

Đây là demo tương tác nhỏ, chưa phải ứng dụng Streamlit. Thay `demo_text` rồi chạy lại cell.
Điểm Softmax không có nghĩa mô hình chắc chắn đúng với tỷ lệ phần trăm đó.

In [ ]:
def predict_comment(text):
    clean_text = normalize_text(text)
    if not clean_text:
        raise ValueError("Hãy nhập một bình luận không rỗng.")
    if len(clean_text.split()) > CONFIG["MAX_LENGTH"]:
        print(f"Lưu ý: chỉ dùng {CONFIG['MAX_LENGTH']} token đầu câu.")
    scores = loaded_model(tf.constant([clean_text]), training=False).numpy()[0]
    predicted_label = int(scores.argmax())
    print("Nhãn dự đoán:", LABEL_VI[predicted_label])
    return pd.DataFrame({"Nhãn": LABEL_VI, "Điểm Softmax": scores})

demo_text = "Giảng viên giải thích rất dễ hiểu, mình học được nhiều điều."
display(predict_comment(demo_text))


## 12. Tải kết quả và chuẩn bị báo cáo

Lưu một bộ artifact cho lần train được chọn. ZIP gồm model, số liệu, hình và mã suy luận;
không chứa ZIP dataset gốc. CSV dự đoán có văn bản test: kiểm tra điều kiện chia sẻ trước
khi đưa toàn bộ artifact lên thư mục công khai.

Trong báo cáo, ghi đúng subset, số epoch thực chạy, GPU thực tế, thời gian train,
Macro-F1 và F1 từng lớp. Không báo một tỷ lệ giảm vanishing gradient chưa đo.

In [ ]:
archive_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("File kết quả:", archive_path)
try:
    from google.colab import files
except ImportError:
    print("Đang chạy ngoài Colab; mở file ZIP tại đường dẫn trên để lấy kết quả.")
else:
    files.download(archive_path)
